In [20]:
import tonemapper
import torch as th
import numpy as np
from sh_utils import get_shcoeff, flatten_sh_coeff, unfold_sh_coeff, apply_integrate_conv_anyLmax, sample_from_sh, cartesian_to_spherical, genSurfaceNormals
from sh_utils_vectorized import unfold_sh_coeff_vec, flatten_sh_coeff_vec
import skimage
import time
tonemapper = tonemapper.TonemapHDR()
def render(hdr_image, face, Lmax):
    hdr_tm, _, _ = tonemapper(hdr_image)

    coeff = get_shcoeff(hdr_image, Lmax=Lmax)   # 3, 2, Lmax+1, Lmax+1
    sh = flatten_sh_coeff(coeff, max_sh_level=Lmax) # 3, (Lmax+1)^2
    unfolded = unfold_sh_coeff(sh, max_sh_level=Lmax)   # 3, 2, Lmax+1, Lmax+1

    apply_integrated = apply_integrate_conv_anyLmax(unfolded.copy(), Lmax)

    if face is None:
        normal_map_org, mask = genSurfaceNormals(256)  # H, W, C
        normal_map_org = normal_map_org.permute(1, 2, 0).cpu().numpy()
        normal_map = normal_map_org.copy()
        mask = mask.cpu().numpy()[..., None]
        T = th.tensor([[0.,0.,1.],
                        [1.,0.,0.],
                        [0.,1.,0.]])                       # maps [x,y,z] -> [z,x,y]
        normal_map = th.einsum('ij,hwj->hwi', T, th.tensor(normal_map).float()).cpu().numpy()
        normal_map = normal_map
    else:
        normal_map_org = face['normal_map']
        normal_map = normal_map_org.copy()
        mask = face['alpha_map']
        T = th.tensor([[0.,0.,1.],
                        [1.,0.,0.],
                        [0.,1.,0.]])                       # maps [x,y,z] -> [z,x,y]
        normal_map = th.einsum('ij,hwj->hwi', T, th.tensor(normal_map).float()).cpu().numpy()
        normal_map = normal_map
    

    theta, phi = cartesian_to_spherical(normal_map)
    shading = sample_from_sh(apply_integrated, lmax=Lmax, theta=theta, phi=phi)
    if face is not None:
        shading = shading * face['albedo']

    shading = np.float32(shading)

    return ((normal_map_org + 1) * 0.5), ((normal_map + 1) * 0.5), shading, mask, coeff, sh, unfolded

In [21]:

Lmax = 2
n_frames = 60
hdr = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/neural_gaffer_environment_map_sample/012_hdrmaps_com_free_2K.exr"
hdr_image = skimage.io.imread(hdr).astype(np.float32)  # H, W, 3

time_start = time.time()
render(hdr_image, face=None, Lmax=Lmax)
time_end = time.time()
print(f"Default test took {time_end - time_start:.4f} seconds")

time_start = time.time()
coeff_list_v, sh_list_v, unfolded_list_v = test_vect(hdr_image, Lmax=Lmax, n_frames=n_frames)
time_end = time.time()
print(f"Vectorized test took {time_end - time_start:.4f} seconds")


/home/mint/miniconda3/envs/dpm_sampling_deca_pysh_env/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3526.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Default test took 5.8399 seconds
Vectorized test took 2.6433 seconds


In [ ]:
import torch as th
import numpy as np
from sh_utils import get_shcoeff, flatten_sh_coeff, unfold_sh_coeff, apply_integrate_conv_anyLmax
from sh_utils_vectorized import unfold_sh_coeff_vec, flatten_sh_coeff_vec
import skimage
import time

Lmax = 2
n_frames = 60
hdr = "/home/mint/Dev/DiFaReli/difareli-faster/experiment_scripts/TPAMI/neural_gaffer_environment_map_sample/012_hdrmaps_com_free_2K.exr"
hdr_image = skimage.io.imread(hdr).astype(np.float32)  # H, W, 3

def test_default(hdr_image, Lmax=2, n_frames=60):
    coeff_list = []
    sh_list = []
    unfolded_list = []
    for i in range(n_frames):
        coeff = get_shcoeff(hdr_image, Lmax=Lmax)   # 3, 2, Lmax+1, Lmax+1
        sh = flatten_sh_coeff(coeff, max_sh_level=Lmax) # 3, (Lmax+1)^2
        unfolded = unfold_sh_coeff(sh, max_sh_level=Lmax)   # 3, 2, Lmax+1, Lmax+1
        apply_integrated = apply_integrate_conv_anyLmax(unfolded.copy(), Lmax)
        coeff_list.append(coeff)
        sh_list.append(sh)
        unfolded_list.append(unfolded)
    return coeff_list, sh_list, unfolded_list

def test_vect(hdr_image, Lmax=2, n_frames=60):
    coeff_list = []
    sh_list = []
    unfolded_list = []
    for i in range(n_frames):
        coeff = get_shcoeff(hdr_image, Lmax=Lmax)   # 3, 2, Lmax+1, Lmax+1
        sh = flatten_sh_coeff_vec(coeff, max_sh_level=Lmax) # 3, (Lmax+1)^2
        unfolded = unfold_sh_coeff_vec(sh, max_sh_level=Lmax)   # 3, 2, Lmax+1, Lmax+1
        coeff_list.append(coeff)
        sh_list.append(sh)
        unfolded_list.append(unfolded)
    return coeff_list, sh_list, unfolded_list


time_start = time.time()
coeff_list, sh_list, unfolded_list = test_default(hdr_image, Lmax=Lmax, n_frames=n_frames)
time_end = time.time()
print(f"Default test took {time_end - time_start:.4f} seconds")

time_start = time.time()
coeff_list_v, sh_list_v, unfolded_list_v = test_vect(hdr_image, Lmax=Lmax, n_frames=n_frames)
time_end = time.time()
print(f"Vectorized test took {time_end - time_start:.4f} seconds")


Default test took 2.5244 seconds
Vectorized test took 2.5228 seconds
